# Evaluating and Optimizing LLM Agents

**D. Brian Letort, PhD** — Head of Data Office & Platform AI, Author, & Adjunct Professor

This notebook covers two modules:
1. **Measuring Agent Quality** — why and what to measure, DeepEval live scoring, LLM-as-a-Judge (G-Eval)
2. **From Custom Metrics to Production Dashboards** — custom metrics with Open-RAG-Eval, the observability triad, LangSmith

---

## Part 1 — Measuring Agent Quality

---

### Why Evaluate LLM Agents?

**Trust = Quality = Adoption**

- LLMs are **probabilistic**: even great prompts yield variation between runs
- You can't optimize what you don't measure
- Hallucinations and irrelevant answers destroy credibility
- Evaluation enables safe iteration, tuning, and monitoring

The three pillars that evaluation protects:

| Pillar | Question |
|--------|----------|
| Quality | Are responses accurate and helpful? |
| Trust | Can users rely on the agent? |
| ROI | Is the investment paying off? |

---

### What Should You Measure?

**Core quality dimensions:**

| Dimension | Question |
|-----------|----------|
| **Relevance** | Does it answer the user's question? |
| **Correctness** | Is it factually accurate? |
| **Hallucination Rate** | How often does it make stuff up? |
| **Contextual Fit** | Is the answer supported by context? |

**Measurement approaches:**

| Method | Description | Notes |
|--------|-------------|-------|
| Traditional (BLEU, ROUGE, F1) | Token overlap with reference | Fast, no LLM needed, but misses semantics |
| Semantic (Cosine similarity) | Embedding-space similarity | Better than token overlap, still imperfect |
| **LLM-as-a-Judge (G-Eval)** | Role-played grading by an LLM | No gold label needed; justified, scored, explainable |

---

### From Measurement to Action — The 4-Step Workflow

```
Step 1               Step 2                  Step 3                Step 4
──────────────       ──────────────────────  ──────────────────    ──────────────────────
Start with a    ──►  Integrate live     ──►  Add judgement    ──►  Upload docs, run
RAG Agent            scoring with            scoring with          queries, observe
                     DeepEval                G-Eval
```

**Outcome:** Real-time feedback loop → higher quality, faster tuning

---

## Demo 1 — DeepEval Live Scoring

Adding DeepEval for live relevance & hallucination scoring on a RAG agent.

### Setup

In [ ]:
# %pip install deepeval langchain langchain-openai langchain-community openai faiss-cpu

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
# Expects OPENAI_API_KEY in environment

### A Minimal RAG Agent

In [ ]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains import RetrievalQA

# Sample documents (replace with your own corpus)
documents = [
    "LangChain is a framework for building LLM-powered applications.",
    "RAG stands for Retrieval-Augmented Generation. It grounds LLM answers in external documents.",
    "FAISS is a library for efficient similarity search over dense vectors.",
    "Hallucination in LLMs refers to generating plausible-sounding but factually incorrect content.",
    "DeepEval is an open-source LLM evaluation framework supporting metrics like relevancy and hallucination.",
]

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
chunks = splitter.create_documents(documents)

embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(chunks, embeddings)

llm = ChatOpenAI(model="gpt-4o", temperature=0)
rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(search_kwargs={"k": 2}),
    return_source_documents=True,
)

print("RAG agent ready.")

### Integrating DeepEval Metrics

In [ ]:
from deepeval.metrics import AnswerRelevancyMetric, HallucinationMetric
from deepeval.test_case import LLMTestCase

def run_with_eval(question: str) -> dict:
    """Run the RAG agent and score the response with DeepEval."""
    result = rag_chain.invoke({"query": question})
    answer = result["result"]
    context = [doc.page_content for doc in result["source_documents"]]

    test_case = LLMTestCase(
        input=question,
        actual_output=answer,
        retrieval_context=context,
    )

    relevancy_metric = AnswerRelevancyMetric(threshold=0.7, model="gpt-4o")
    hallucination_metric = HallucinationMetric(threshold=0.5, model="gpt-4o")

    relevancy_metric.measure(test_case)
    hallucination_metric.measure(test_case)

    return {
        "question": question,
        "answer": answer,
        "relevancy_score": relevancy_metric.score,
        "relevancy_reason": relevancy_metric.reason,
        "hallucination_score": hallucination_metric.score,
        "hallucination_reason": hallucination_metric.reason,
    }


# Try it
result = run_with_eval("What is RAG?")
print(f"Answer: {result['answer']}")
print(f"Relevancy: {result['relevancy_score']:.2f} — {result['relevancy_reason']}")
print(f"Hallucination: {result['hallucination_score']:.2f} — {result['hallucination_reason']}")

### Batch Evaluation

In [ ]:
import pandas as pd

test_questions = [
    "What is LangChain used for?",
    "What does RAG stand for?",
    "What is hallucination in the context of LLMs?",
    "Who invented the internet?",  # out-of-scope — expect low relevancy
]

rows = []
for q in test_questions:
    try:
        rows.append(run_with_eval(q))
    except Exception as e:
        print(f"Error on '{q}': {e}")

df = pd.DataFrame(rows)[["question", "relevancy_score", "hallucination_score", "answer"]]
df

---

## Part 2 — Wiring in LLM-as-a-Judge (G-Eval)

### Prompt Structure

| Field | Value |
|-------|-------|
| **Role** | Impartial evaluator ("You are a judge…") |
| **Input** | User question |
| **Context** | Grounding text (~1 500 characters) |
| **Answer** | LLM agent's response |
| **Scoring scale** | 1–10 |
| **Rationale** | 1-sentence explanation |
| **Temperature** | Always **0** for consistency |

### Single Judge vs. Multi-Judge

| Approach | Pros | Cons |
|----------|------|------|
| **Single Judge** | Fast, low cost | Sensitive to prompt variance |
| **Multi-Judge** | Stable, captures scoring variance | More expensive, more complex logic |

> Use **majority vote** or **averaging** to reduce prompt bias and improve reliability.

### Making Evaluations Actionable

The pattern for integrating a judge into any pipeline:

1. **Wrap Response with `judge()` call** — call the judge LLM with context, question, and answer
2. **Score + Rationale returned** — LLM gives a numeric score plus a one-liner explanation
3. **Display in sidebar UI** — show relevance, hallucination score, and justification alongside the answer

In [ ]:
from openai import OpenAI
import json

client = OpenAI()

JUDGE_SYSTEM_PROMPT = """\
You are an impartial evaluator assessing the quality of an AI assistant's answer.
You will be given:
  - A user question
  - Context documents that the assistant had access to
  - The assistant's answer

Evaluate on two dimensions and respond with valid JSON only:
{
  "relevancy_score": <1-10>,
  "relevancy_rationale": "<one sentence>",
  "hallucination_score": <1-10>,
  "hallucination_rationale": "<one sentence>"
}

Scoring guide:
  relevancy_score 10 = perfectly answers the question; 1 = completely off-topic
  hallucination_score 1 = heavily hallucinated; 10 = fully grounded in context
"""


def judge(question: str, context: list[str], answer: str) -> dict:
    context_text = "\n---\n".join(context)[:1500]
    user_message = f"""Question: {question}

Context:
{context_text}

Answer: {answer}"""

    response = client.chat.completions.create(
        model="gpt-4o",
        temperature=0,
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
            {"role": "user", "content": user_message},
        ],
    )
    return json.loads(response.choices[0].message.content)


# Demo
test_q = "What is DeepEval?"
rag_result = rag_chain.invoke({"query": test_q})
scores = judge(
    question=test_q,
    context=[d.page_content for d in rag_result["source_documents"]],
    answer=rag_result["result"],
)
print(json.dumps(scores, indent=2))

### Multi-Judge Averaging

In [ ]:
def multi_judge(question: str, context: list[str], answer: str, n: int = 3) -> dict:
    """Run the judge n times and average the scores for stability."""
    results = [judge(question, context, answer) for _ in range(n)]
    avg_relevancy = sum(r["relevancy_score"] for r in results) / n
    avg_hallucination = sum(r["hallucination_score"] for r in results) / n
    return {
        "avg_relevancy_score": round(avg_relevancy, 2),
        "avg_hallucination_score": round(avg_hallucination, 2),
        "individual_results": results,
    }


multi_scores = multi_judge(
    question=test_q,
    context=[d.page_content for d in rag_result["source_documents"]],
    answer=rag_result["result"],
    n=3,
)
print(f"Avg Relevancy: {multi_scores['avg_relevancy_score']}")
print(f"Avg Hallucination: {multi_scores['avg_hallucination_score']}")

---

## Part 3 — From Custom Metrics to Production Dashboards

---

### When Generic Metrics Fail

Standard relevancy and hallucination scores don't capture domain-specific requirements:

| Gap | Example |
|-----|--------|
| **Compliance requirements** | Regulated industries need specific disclaimers or formats |
| **Citation fidelity** | Legal/medical agents must cite sources explicitly |
| **PII exposure risk** | Agent must never surface personally identifiable information |

---

### The Open-RAG-Eval Toolkit

**Capabilities:**
- Synthetic question-answer pair generation
- Multi-metric evaluation runner
- Integration-friendly YAML configuration

**Common use-cases:**
- Robustness checks
- Edge-case handling
- Compliance verification

### Stress Scenarios to Test

| Scenario | Description |
|----------|-------------|
| **Ambiguous queries** | Multiple plausible answers — which does the agent choose? |
| **Noisy input** | Typos, jargon, slang — does the agent still understand intent? |
| **Adversarial attempts** | Prompt injection, misleading context — is the agent robust? |

---

## Demo 2 — Custom Metric: Citation Accuracy

A simple custom metric that checks whether the agent's response includes a source citation.

In [ ]:
def citation_accuracy_score(prediction: str) -> int:
    """Returns 1 if the response contains a source citation, 0 otherwise."""
    return int("Source:" in prediction)


# Test samples
sample_predictions = [
    "RAG combines retrieval and generation. Source: LangChain docs.",
    "I'm not sure about that topic.",
    "The capital of France is Paris. Source: Wikipedia.",
    "This answer has no citation at all.",
    "Hallucination is a known LLM issue. Source: Anthropic research blog.",
]

passed = sum(citation_accuracy_score(p) for p in sample_predictions)
total = len(sample_predictions)
print(f"Citation Accuracy: {passed}/{total} = {passed/total:.0%}")

### Custom Metric as a DeepEval-compatible class

In [ ]:
from deepeval.metrics import BaseMetric
from deepeval.test_case import LLMTestCase


class CitationAccuracyMetric(BaseMetric):
    """Passes when the agent response contains 'Source:' attribution."""

    def __init__(self, threshold: float = 1.0):
        self.threshold = threshold

    @property
    def __name__(self):
        return "Citation Accuracy"

    def measure(self, test_case: LLMTestCase) -> float:
        self.score = float(citation_accuracy_score(test_case.actual_output))
        self.success = self.score >= self.threshold
        self.reason = "Response includes source citation." if self.success else "No 'Source:' found in response."
        return self.score

    async def a_measure(self, test_case: LLMTestCase) -> float:
        return self.measure(test_case)

    def is_successful(self) -> bool:
        return self.success


# Verify it works
metric = CitationAccuracyMetric()
tc = LLMTestCase(input="What is RAG?", actual_output="RAG grounds answers in docs. Source: paper.")
metric.measure(tc)
print(f"Score: {metric.score}, Passed: {metric.success}, Reason: {metric.reason}")

### Multi-Metric Evaluation Run

In [ ]:
import pandas as pd


def evaluate_batch(test_cases: list[dict]) -> pd.DataFrame:
    """
    Run answer_relevancy, hallucination, and citation_accuracy over a list of
    {input, actual_output, retrieval_context} dicts.
    """
    rows = []
    for tc_dict in test_cases:
        tc = LLMTestCase(
            input=tc_dict["input"],
            actual_output=tc_dict["actual_output"],
            retrieval_context=tc_dict.get("retrieval_context", []),
        )
        rel = AnswerRelevancyMetric(threshold=0.7, model="gpt-4o")
        hal = HallucinationMetric(threshold=0.5, model="gpt-4o")
        cit = CitationAccuracyMetric()

        rel.measure(tc)
        hal.measure(tc)
        cit.measure(tc)

        rows.append({
            "input": tc_dict["input"],
            "answer_relevancy": rel.score,
            "hallucination": hal.score,
            "citation_accuracy": cit.score,
            "output": tc_dict["actual_output"][:80] + "...",
        })
    return pd.DataFrame(rows)


test_cases = [
    {
        "input": "What is LangChain?",
        "actual_output": "LangChain is a framework for LLM applications. Source: LangChain docs.",
        "retrieval_context": ["LangChain is a framework for building LLM-powered applications."],
    },
    {
        "input": "What is FAISS?",
        "actual_output": "FAISS is used for similarity search.",
        "retrieval_context": ["FAISS is a library for efficient similarity search over dense vectors."],
    },
    {
        "input": "Who won the 2022 World Cup?",
        "actual_output": "Argentina won the 2022 FIFA World Cup.",
        "retrieval_context": ["LangChain is a framework for building LLM-powered applications."],
    },
]

results_df = evaluate_batch(test_cases)
results_df

---

## Part 4 — Holistic Cost, Latency, and Quality Tuning

---

### The Observability Triad

```
              Cost
             /    \
            /      \
        Quality ── Latency
```

| Dimension | Question |
|-----------|----------|
| **Quality** | Are the answers accurate and helpful? |
| **Cost** | How many tokens are used per response? Is it sustainable at scale? |
| **Latency** | How fast are user questions answered? Does it feel instant or sluggish? |

All three interact: reducing context length lowers cost and latency but may hurt quality.

---

### LangSmith — Production-Ready LLM Observability

> Purpose-built for LLM agents: **logs, traces, costs, and quality** in one place

| Feature | What it does |
|---------|-------------|
| **Trace capture** | Follows each prompt through the full pipeline |
| **Judgement metrics** | Automated scoring, custom evaluation |
| **Dashboards** | At-a-glance cost and latency breakdown |

## Demo 3 — LangSmith Tracing

Instrument the RAG chain with LangSmith to capture traces and observe cost/latency.

In [ ]:
# %pip install langsmith

In [ ]:
import os

# Set LangSmith environment variables before importing LangChain
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "llm-agent-evaluation"
# os.environ["LANGCHAIN_API_KEY"] = "your-langsmith-api-key"  # or set in .env

print("LangSmith tracing enabled — all LangChain calls will be traced.")

In [ ]:
import time

def run_traced(question: str) -> dict:
    """Run the RAG chain and capture latency alongside quality scores."""
    t0 = time.perf_counter()
    result = rag_chain.invoke({"query": question})
    latency_ms = (time.perf_counter() - t0) * 1000

    scores = judge(
        question=question,
        context=[d.page_content for d in result["source_documents"]],
        answer=result["result"],
    )

    return {
        "question": question,
        "latency_ms": round(latency_ms),
        **scores,
        "answer": result["result"],
    }


traced_result = run_traced("What is hallucination in LLMs?")
print(f"Latency: {traced_result['latency_ms']} ms")
print(f"Relevancy: {traced_result['relevancy_score']}/10 — {traced_result['relevancy_rationale']}")
print(f"Hallucination: {traced_result['hallucination_score']}/10 — {traced_result['hallucination_rationale']}")

### Visualising the Triad Over Multiple Queries

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Simulate a batch run (use run_traced() in practice)
np.random.seed(42)
n = 20
simulated = pd.DataFrame({
    "query_id": range(1, n + 1),
    "relevancy_score": np.random.uniform(5, 10, n),
    "hallucination_score": np.random.uniform(6, 10, n),
    "latency_ms": np.random.randint(300, 1800, n),
    "prompt_tokens": np.random.randint(200, 800, n),
})

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].bar(simulated["query_id"], simulated["relevancy_score"], color="steelblue")
axes[0].axhline(7, color="red", linestyle="--", label="threshold")
axes[0].set_title("Relevancy Score per Query")
axes[0].set_xlabel("Query ID")
axes[0].set_ylabel("Score (1–10)")
axes[0].legend()

axes[1].bar(simulated["query_id"], simulated["latency_ms"], color="coral")
axes[1].set_title("Latency per Query (ms)")
axes[1].set_xlabel("Query ID")
axes[1].set_ylabel("ms")

axes[2].scatter(simulated["prompt_tokens"], simulated["latency_ms"], c=simulated["relevancy_score"],
                cmap="RdYlGn", vmin=1, vmax=10, s=80, edgecolors="k", linewidths=0.5)
axes[2].set_title("Cost vs Latency (colour = relevancy)")
axes[2].set_xlabel("Prompt Tokens (proxy for cost)")
axes[2].set_ylabel("Latency (ms)")
plt.colorbar(axes[2].collections[0], ax=axes[2], label="Relevancy")

plt.tight_layout()
plt.show()

---

## Summary

| Topic | Key Takeaway |
|-------|-------------|
| **Why evaluate** | Trust = Quality = Adoption; LLMs are probabilistic |
| **What to measure** | Relevance, Correctness, Hallucination Rate, Contextual Fit |
| **DeepEval** | Drop-in metrics for RAG pipelines; real-time scoring |
| **LLM-as-a-Judge (G-Eval)** | Role-played grading; no gold labels needed; always use temperature=0 |
| **Custom metrics** | Use when compliance, citation, or PII requirements go beyond generic scores |
| **Open-RAG-Eval** | Synthetic QA generation + YAML-driven multi-metric runner |
| **Observability triad** | Quality, Cost, Latency — all three must be tracked in production |
| **LangSmith** | Production traces, judgement metrics, and dashboards for LangChain apps |

### Next Steps

1. Run `evaluate_batch()` on your own RAG corpus
2. Add `CitationAccuracyMetric` (or a domain-specific variant) to your eval suite
3. Enable LangSmith tracing in your app and build a cost/latency/quality dashboard
4. Set up regression alerts: fail CI if any metric drops below its threshold

---
*Module 04 — Agentic AI for Developers*